# Wikipedia Retriever

In [4]:
from langchain_community.retrievers import WikipediaRetriever

In [5]:
retriever = WikipediaRetriever(top_k_results=2, lang='en')
query= 'Who is APJ abdul kalam'

retriever.invoke(query)

[Document(metadata={'title': 'A. P. J. Abdul Kalam', 'summary': 'Avul Pakir Jainulabdeen Abdul Kalam (  UB-duul kə-LAHM; 15 October 1931 – 27 July 2015) was an Indian aerospace scientist and statesman who served as the  president of India from 2002 to 2007.\nBorn and raised in a Muslim family in Rameswaram, Tamil Nadu, Kalam studied physics and aerospace engineering. He spent the next four decades as a scientist and science administrator, mainly at the Defence Research and Development Organisation (DRDO) and Indian Space Research Organisation (ISRO) and was intimately involved in India\'s civilian space programme and military missile development efforts. He was known as the "Missile Man of India" for his work on the development of ballistic missile and launch vehicle technology. He also played a pivotal organisational, technical, and political role in Pokhran-II nuclear tests in 1998, India\'s second such test after the first test in 1974.\nKalam was elected as the president of India i

# Vector Store retriever

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [7]:
doc = [
    Document(page_content='Black holes are regions of spacetime where gravity is so strong that nothing, including light or other electromagnetic waves, has enough energy to escape it. The boundary of no escape is called the event horizon.'),
    Document(page_content='To make authentic Italian carbonara, you need guanciale, Pecorino Romano cheese, eggs, and black pepper. Unlike many international variations, the traditional recipe strictly forbids the use of cream or garlic.'),
    Document(page_content='The library of Alexandria was one of the largest and most significant libraries of the ancient world. Dedicated to the Muses, the nine goddesses of the arts, it flourished under the patronage of the Ptolemaic dynasty in Egypt.'),
    Document(page_content='Quantum superposition is a fundamental principle of quantum mechanics. It states that, much like waves in classical physics, any two (or more) quantum states can be added together (superposed) and the result will be another valid quantum state.'),
    Document(page_content='Photosynthesis is the process used by plants, algae, and certain bacteria to harness energy from sunlight and turn it into chemical energy. This energy is stored in carbohydrate molecules, such as sugars, which are synthesized from carbon dioxide and water.')
]

In [8]:
from dotenv import load_dotenv
load_dotenv()

vector_store = Chroma.from_documents(
    documents=doc,
    embedding=GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001'),
    collection_name="retriever"
)

In [9]:
retriever = vector_store.as_retriever(search_kwargs ={'k':2})

query = "How do plants grow?"
result = retriever.invoke(query)
print(result)

[Document(metadata={}, page_content='Photosynthesis is the process used by plants, algae, and certain bacteria to harness energy from sunlight and turn it into chemical energy. This energy is stored in carbohydrate molecules, such as sugars, which are synthesized from carbon dioxide and water.'), Document(metadata={}, page_content='Quantum superposition is a fundamental principle of quantum mechanics. It states that, much like waves in classical physics, any two (or more) quantum states can be added together (superposed) and the result will be another valid quantum state.')]


# MMR (Maximal Marginal Relevance)

In [10]:
documents = [
    # Document 1: General Definition (The "Anchor")
    Document("Large Language Models (LLMs) are advanced artificial intelligence systems designed to understand, generate, and manipulate human language. They are built using deep learning techniques."),
    
    # Document 2: Highly Redundant (MMR should filter this out)
    Document("LLMs are AI models that can process and produce text. They utilize deep learning to perform tasks like understanding and generating human-like language."),

    # Document 3: Distinct Aspect - Training (MMR should include this)
    Document("Training these models requires massive datasets scraped from the internet, books, and articles, along with significant computational power from GPU clusters."),

    # Document 4: Distinct Aspect - Limitations (MMR should include this)
    Document("A major challenge with LLMs is 'hallucination', a phenomenon where the model confidently generates false or nonsensical information that is not grounded in fact."),

    # Document 5: Distinct Aspect - Applications (MMR should include this)
    Document("Applications of LLMs are vast, ranging from customer service chatbots and code generation assistants to automated language translation and creative writing tools.")
]

In [11]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001'),
    collection_name='MMR_retriever'
)

In [12]:
MMR_retriever = vector_store.as_retriever(search_type='mmr',
                                          search_kwargs={'k':2})

query = "What is a Large Language Model?"
MMR_retriever.invoke(query)


[Document(metadata={}, page_content='Large Language Models (LLMs) are advanced artificial intelligence systems designed to understand, generate, and manipulate human language. They are built using deep learning techniques.'),
 Document(metadata={}, page_content='Training these models requires massive datasets scraped from the internet, books, and articles, along with significant computational power from GPU clusters.')]

# Multi Query Retriever

In [13]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

ModuleNotFoundError: No module named 'langchain_community.retrievers.azure_cognitive_search'

In [14]:
documents =[
    
    # 1. Propulsion Systems
    Document("Chemical rockets function by burning a fuel and an oxidizer in a combustion chamber. The hot gases are expelled through a nozzle at high speeds, generating thrust according to Newton's third law of motion."),
    # 2. Mars Colonization
    Document(
    "The colonization of Mars faces significant challenges, primarily radiation exposure and the lack of a breathable atmosphere. Proposed solutions include underground habitats and producing oxygen from the Martian atmosphere using MOXIE technology."),
    # 3. Space Debris (The "Junk" Problem)
    Document(
    "Orbital debris, or 'space junk', poses a severe threat to active satellites and spacecraft. Even a small paint fleck traveling at orbital velocities (17,500 mph) can cause catastrophic damage upon impact."),
    # 4. The International Space Station (ISS)
    Document(
    "The ISS is a modular space station in low Earth orbit. It serves as a microgravity and space environment research laboratory in which crew members conduct experiments in biology, human biology, physics, astronomy, and meteorology."),
    # 5. Telescopes (James Webb)
    Document(
    "The James Webb Space Telescope (JWST) operates primarily in the infrared spectrum. This allows it to see through dust clouds where stars are being born and observe the light from the very first galaxies formed after the Big Bang."),
    # 6. Exoplanets
    Document(
    "Astronomers discover exoplanets—planets orbiting other stars—using the transit method. This involves measuring the tiny dip in a star's brightness as a planet passes in front of it, blocking a small fraction of the light."),
    # 7. Space Suits (EVA)
    Document(
    "Extravehicular Mobility Units (EMUs), or space suits, are essentially miniature personal spacecraft. They provide oxygen, temperature regulation, and protection from micrometeoroids and cosmic radiation during spacewalks."),
    # 8. The Artemis Program
    Document(
    "NASA's Artemis program aims to return humans to the Moon, specifically the lunar south pole. The long-term goal is to establish a sustainable presence on the Moon to prepare for future crewed missions to Mars."),
    # 9. Satellite Communication
    Document(
    "Geostationary satellites orbit the Earth at the same speed the planet rotates, appearing fixed in the sky. This orbit is ideal for telecommunications and weather monitoring because ground antennas do not need to move to track them."),
    # 10. Deep Space Probes (Voyager)
    Document(
    "The Voyager probes, launched in 1977, are the farthest human-made objects from Earth. They carry the 'Golden Record', a phonograph record containing sounds and images selected to portray the diversity of life and culture on Earth.")
]


In [15]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001"),
    collection_name='MultiQueryRetriever'
)

In [16]:
MultiQuery_retriever = MultiQueryRetriever(
    retriever= vector_store.as_retriever(search_kwargs ={'k':2}),
    llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
)

NameError: name 'MultiQueryRetriever' is not defined

In [17]:
query = "How do humans explore and survive in space?"

In [18]:
MultiQuery_retriever.invoke(query)

NameError: name 'MultiQuery_retriever' is not defined

# ContextualCompressionRetriever

In [5]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

ModuleNotFoundError: No module named 'langchain_community.retrievers.azure_cognitive_search'